# Swin-T + Mask R-CNN (COCO) — DIMER object-detection tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/swin-detection-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/swin-detection-pipeline/blob/main/tutorials/swin_detection_task_inference.ipynb) [![Python 3.10 required](https://img.shields.io/badge/Python-3.10%20required-3776ab?style=flat&logo=python&logoColor=white)](https://github.com/kurtvalcorza/swin-detection-pipeline/blob/main/README.md) [![Checkpoint](https://img.shields.io/badge/OpenMMLab-mask--rcnn__swin--t--p4--w7__fpn__1x__coco-ffcc4d?style=flat)](https://github.com/open-mmlab/mmdetection/tree/v3.3.0/configs/swin) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2FSwin--Transformer-181717?style=flat&logo=github&logoColor=white)](https://github.com/microsoft/Swin-Transformer) [![arXiv](https://img.shields.io/badge/arXiv-2103.14030-b31b1b.svg)](https://arxiv.org/abs/2103.14030)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** pretrained COCO-80 object detection (class-labelled axis-aligned boxes with uncalibrated scores) using the pinned OpenMMLab `mask-rcnn_swin-t-p4-w7_fpn_1x_coco` checkpoint through the repository's `DimerSwinDetector` API

**This notebook is standalone.** It carries the repository's package (2 modules under `src/dimer_swin_detection/`, at revision `fd4ac01639b0`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the OpenMMLab checkpoint host (`download.openmmlab.com`) at the immutable MMDetection release-tag commit `44ebd17b145c2372c4b700bfb9cb20dbd28ab64a` (~191 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

At inference the pinned Swin-T backbone + Mask R-CNN head maps one RGB image to class-labelled boxes and class scores over the 80 COCO 2017 categories; the DIMER v1 contract exposes boxes, classes and scores only (the checkpoint's instance masks are not part of the public API). **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream OpenMMLab checkpoint supplies the weights and the pinned MMDetection 3.3.0 package supplies the config and test pipeline, and the carried package adds snapshot verification, runtime-version checks, input validation, a fixed output contract and the `coco_box_ap`, `validate_inputs` and `evaluation_report` helpers.

**Trust boundary (MOD12).** The checkpoint is a code-capable PyTorch `.pth` serialization. The carried `verify_snapshot` re-hashes it against the inline manifest (and against the digest the package has always pinned in `MODEL_SPEC`) before the pinned MMDetection loader deserializes it inside `mmengine`; the deserialization call is upstream's, not the package's, and is **not** a `weights_only` load. A matching digest proves byte identity with the pinned OpenMMLab distribution, not publisher authenticity — run this notebook only where that pinned source is trusted.

The default sample is a synthetic scene generated in code (no download, no ground truth), so its detections are demonstration (plumbing) evidence, not a correctness or benchmark claim. A gated option fetches the public labelled COCO8 validation subset instead and evaluates COCO AP on it.

**Learning objectives:** install the pinned Python 3.10 OpenMMLab CPU runtime, read what the carried package guarantees, resolve and digest-verify the immutable OpenMMLab checkpoint, generate a synthetic default input (or opt into the labelled COCO8 subset / a BYOD image) and validate it into an input manifest, run detection through the public API, read uncalibrated class scores and the caller-owned `score_threshold` correctly, produce an evaluation report that is `sample-sanity` with `coco_box_ap` only when ground-truth boxes exist and `not-measurable` otherwise, and export machine-readable detections plus provenance.

**This notebook does not demonstrate:** instance or semantic segmentation (the checkpoint's mask head is outside the DIMER v1 contract), open-vocabulary detection, tracking, keypoints, image classification, or any training or fine-tuning. The label space is fixed to the 80 COCO 2017 categories; objects outside that space are either missed or assigned a COCO label.

## Prerequisites

- **Runtime:** a **CPython 3.10** Jupyter kernel on Linux (the notebook asserts `sys.version_info[:2] == (3, 10)` and stops otherwise). The qualified OpenMMLab stack — torch 2.1.2 (CPU build), MMCV 2.1.0, MMEngine 0.10.7, MMDetection 3.3.0, NumPy 1.26.4 — has prebuilt wheels for Python 3.10 only; `pip` cannot change the interpreter, so a Python 3.11+ kernel (including current default Colab runtimes) is unsupported and the pinned install fails there. CPU is the default and only qualified path; no GPU is required. The pinned torch/mmcv wheels are the largest downloads of the run.
- **Knowledge:** basic Python and image handling; what a detection score and an IoU-based AP metric are.
- **Data:** the default sample is a deterministic 640×480 synthetic scene (gradient background plus flat-coloured shapes) generated in code, so nothing is downloaded and there is no ground truth. Two optional gates are off by default so the sample path runs top-to-bottom without interaction: `USE_COCO8` fetches the public COCO8 validation subset (4 labelled COCO 2017 images, a 443 KB archive from the Ultralytics `assets` release `v0.0.0`, verified against its SHA-256 before path-safe extraction) and enables COCO AP evaluation; `USE_BYOD` uploads one image file decodable by Pillow (PNG/JPEG/WebP and similar, at most 64 megapixels). Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the OpenMMLab checkpoint host (`download.openmmlab.com`) only, to fetch the pinned `open-mmlab/mmdetection:mask-rcnn_swin-t-p4-w7_fpn_1x_coco` snapshot (~191 MB in total) at revision `44ebd17b145c…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's tools/pins.txt at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `numpy` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    '--extra-index-url',
    'https://download.pytorch.org/whl/cpu',
    '--find-links',
    'https://download.openmmlab.com/mmcv/dist/cpu/torch2.1/index.html',
    'torch==2.1.2+cpu',
    'torchvision==0.16.2+cpu',
    'mmengine==0.10.7',
    'mmcv==2.1.0',
    'mmdet==3.3.0',
    'pycocotools==2.0.11',
    'numpy==1.26.4',
    'opencv-python==4.10.0.84',
    'pillow==11.3.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'swin-detection-pipeline',
    'repository_revision': 'fd4ac01639b0aa63c2aa0c5cd591cfc4495ad481',
    'embedded_module': 'src/dimer_swin_detection/runtime.py',
    'embedded_modules': ['src/dimer_swin_detection/metrics.py', 'src/dimer_swin_detection/runtime.py'],
    'module_sha256': '92d2e49d5f2bad9ea74da4f6ddcbf1271fc27336176c3235214f1176c4f57db4',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, numpy
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/dimer_swin_detection/` @ `fd4ac01639b0`)

The next 2 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/2:** `src/dimer_swin_detection/metrics.py`

In [ ]:
"""Detection metric helpers carried by the standalone tutorial (NOTEBOOK_SPEC 1.1 EVAL2).

`coco_box_ap` is the repository's only detection metric: COCO AP@[0.50:0.95], AP50 and AP75 over
axis-aligned boxes, computed by `pycocotools` on ground truth the caller supplies. `pycocotools` is
imported lazily so the module (and the notebook cell that carries it) loads without it; the pinned
runtime installs `pycocotools==2.0.11`. `boxes_from_yolo_labels` converts a YOLO-format label file
(normalised `class xc yc w h` rows, as the public COCO8 sample ships) into the ground-truth box shape
`coco_box_ap` consumes. No model logic lives here.
"""

from __future__ import annotations

from collections.abc import Iterable, Mapping
from typing import Any

COCO_NUM_CLASSES = 80  # MMDetection COCO class ordering; COCO category id = class_id + 1 below


def boxes_from_yolo_labels(text: str, width: int, height: int) -> list[dict[str, Any]]:
    """Parse YOLO label rows (`class xc yc w h`, normalised to the image size) into xyxy pixel boxes."""
    if width < 1 or height < 1:
        raise ValueError(f"image size must be positive; got {width}x{height}")
    boxes: list[dict[str, Any]] = []
    for line_number, raw in enumerate(text.splitlines(), start=1):
        line = raw.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"label line {line_number} must have 5 fields (class xc yc w h): {raw!r}")
        class_id = int(parts[0])
        xc, yc, bw, bh = (float(value) for value in parts[1:])
        if not 0 <= class_id < COCO_NUM_CLASSES:
            raise ValueError(
                f"label line {line_number}: class id {class_id} outside 0..{COCO_NUM_CLASSES - 1}"
            )
        w, h = bw * width, bh * height
        x1, y1 = xc * width - w / 2, yc * height - h / 2
        boxes.append({"class_id": class_id, "bbox_xyxy": [x1, y1, x1 + w, y1 + h]})
    return boxes


def _xywh(bbox_xyxy: Iterable[float]) -> list[float]:
    x1, y1, x2, y2 = (float(v) for v in bbox_xyxy)
    if x2 < x1 or y2 < y1:
        raise ValueError(f"bbox_xyxy must satisfy x1<=x2 and y1<=y2: {[x1, y1, x2, y2]}")
    return [x1, y1, x2 - x1, y2 - y1]


def coco_box_ap(
    detections: Iterable[Mapping[str, Any]],
    ground_truth: Iterable[Mapping[str, Any]],
    *,
    num_classes: int = COCO_NUM_CLASSES,
) -> dict[str, Any]:
    """COCO AP@[0.50:0.95], AP50 and AP75 of `detections` against `ground_truth` (pycocotools, bbox).

    `detections`: rows shaped like `Detection.to_dict()` (`image_id`, `class_id`, `score`, `bbox_xyxy`).
    `ground_truth`: one entry per evaluated image,
    `{"image_id": <name>, "boxes": [{"class_id", "bbox_xyxy"}, ...]}`; every detection must name an image
    present in the ground truth. A detector that returns no boxes scores AP 0.0 by construction (the
    empty-detector baseline) and is reported without invoking pycocotools.
    """
    gt_entries = [dict(entry) for entry in ground_truth]
    if not gt_entries:
        raise ValueError("ground_truth must name at least one image")
    image_ids: dict[str, int] = {}
    images = []
    annotations = []
    for entry in gt_entries:
        name = str(entry["image_id"])
        if name in image_ids:
            raise ValueError(f"duplicate ground-truth image_id {name!r}")
        image_ids[name] = len(image_ids) + 1
        images.append({"id": image_ids[name], "file_name": name})
        for box in entry.get("boxes", []):
            class_id = int(box["class_id"])
            if not 0 <= class_id < num_classes:
                raise ValueError(f"ground-truth class id {class_id} outside 0..{num_classes - 1}")
            x, y, w, h = _xywh(box["bbox_xyxy"])
            annotations.append(
                {
                    "id": len(annotations) + 1,
                    "image_id": image_ids[name],
                    "category_id": class_id + 1,
                    "bbox": [x, y, w, h],
                    "area": w * h,
                    "iscrowd": 0,
                }
            )
    results = []
    for row in detections:
        name = str(row["image_id"])
        if name not in image_ids:
            raise ValueError(f"detection names image {name!r} that has no ground-truth entry")
        class_id = int(row["class_id"])
        if not 0 <= class_id < num_classes:
            raise ValueError(f"detection class id {class_id} outside 0..{num_classes - 1}")
        results.append(
            {
                "image_id": image_ids[name],
                "category_id": class_id + 1,
                "bbox": _xywh(row["bbox_xyxy"]),
                "score": float(row["score"]),
            }
        )
    summary = {
        "n_images": len(images),
        "n_ground_truth_boxes": len(annotations),
        "n_detections": len(results),
        "iou_type": "bbox",
    }
    if not results:
        empty = {"coco_ap_50_95": 0.0, "ap50": 0.0, "ap75": 0.0, "note": "no detections: empty-detector AP"}
        return {**summary, **empty}
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    gt = COCO()
    gt.dataset = {
        "info": {"description": "tutorial ground truth"},
        "images": images,
        "annotations": annotations,
        "categories": [{"id": index + 1, "name": str(index)} for index in range(num_classes)],
    }
    gt.createIndex()
    dt = gt.loadRes(results)
    evaluator = COCOeval(gt, dt, "bbox")
    evaluator.params.imgIds = sorted(image_ids.values())
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()
    stats = [float(value) for value in evaluator.stats]
    return {**summary, "coco_ap_50_95": stats[0], "ap50": stats[1], "ap75": stats[2]}

**Module 2/2:** `src/dimer_swin_detection/runtime.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import tempfile
import urllib.request
from collections.abc import Callable, Iterable
from dataclasses import asdict, dataclass
from pathlib import Path

from PIL import Image

# standalone rewrite (build_notebook.py): `from .metrics import coco_box_ap` removed — names are kernel globals defined by the carried modules

MODEL_SPEC = {
    "runtime_id": "swin-t-mask-rcnn-coco-openmmlab-v3.3.0",
    "architecture": "Swin-T + Mask R-CNN",
    "task": "object-detection",
    "dataset": "COCO 2017",
    "mmdetection_version": "3.3.0",
    "mmcv_version": "2.1.0",
    "mmengine_version": "0.10.7",
    "torch_version": "2.1.2",
    "config": "swin/mask-rcnn_swin-t-p4-w7_fpn_1x_coco.py",
    "config_source_revision": "open-mmlab/mmdetection@v3.3.0",
    "checkpoint_url": "https://download.openmmlab.com/mmdetection/v2.0/swin/mask_rcnn_swin-t-p4-w7_fpn_1x_coco/mask_rcnn_swin-t-p4-w7_fpn_1x_coco_20210902_120937-9d6b7cfa.pth",
    "checkpoint_size_bytes": 191461353,
    "checkpoint_sha256": "9d6b7cfaa4aad52ef559611bea454f01d6f1f17c82a1abfac0d71631a193a291",
    "upstream_reported_box_ap": 42.7,
    "instance_masks_in_dimer_contract": False,
}


@dataclass(frozen=True)
class Detection:
    image_id: str
    class_id: int
    class_name: str
    score: float
    bbox_xyxy: tuple[float, float, float, float]

    def to_dict(self) -> dict:
        d = asdict(self)
        d["bbox_xyxy"] = list(self.bbox_xyxy)
        return d


def _sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def _version(dist: str) -> str:
    return importlib.metadata.version(dist)


# ---------------------------------------------------------------------------------------------
# Fleet snapshot scheme (DIMER standalone carrier). The identity constants below name the OpenMMLab
# distribution: MODEL_ID is the config recipe inside the pinned package, MODEL_REVISION the upstream
# git commit of that package's release tag (the config source), and the manifest pins the checkpoint
# bytes. The checkpoint host is download.openmmlab.com, not the Hugging Face Hub, so the staging
# downloader is the pinned URL in MODEL_SPEC rather than hf_hub_download.
# ---------------------------------------------------------------------------------------------
MODEL_ID = "open-mmlab/mmdetection:mask-rcnn_swin-t-p4-w7_fpn_1x_coco"
MODEL_REVISION = "44ebd17b145c2372c4b700bfb9cb20dbd28ab64a"
MODEL_LICENSE = "Apache-2.0"
MODEL_KEY = "swin-t-mask-rcnn-coco"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "mask_rcnn_swin-t-p4-w7_fpn_1x_coco_20210902_120937-9d6b7cfa.pth"
MAX_PIXELS = 64_000_000  # validate_image ceiling


def verify_snapshot(path: str | os.PathLike | None = None) -> dict:
    """Check a local snapshot against its manifest; raise naming the first mismatch.

    The checkpoint entry must also carry the digest MODEL_SPEC has always pinned, so the manifest
    cannot silently re-point the runtime at different bytes.
    """
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    entries = {entry["path"]: entry for entry in manifest.get("files", [])}
    pinned = entries.get(WEIGHTS_FILE)
    if pinned is None or pinned["sha256"] != MODEL_SPEC["checkpoint_sha256"] or pinned["bytes"] != MODEL_SPEC["checkpoint_size_bytes"]:
        raise ValueError(f"manifest entry for {WEIGHTS_FILE} does not match the checkpoint digest pinned in MODEL_SPEC")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _openmmlab_download(relative_path: str, root: Path) -> None:
    """Fetch the pinned OpenMMLab checkpoint into the snapshot directory (the only manifest entry)."""
    if relative_path != WEIGHTS_FILE:
        raise ValueError(f"no pinned download source for {relative_path}")
    target = root / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix="dimer-swin-", suffix=".pth", dir=root)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        with urllib.request.urlopen(MODEL_SPEC["checkpoint_url"], timeout=120) as response, temporary.open("wb") as out:
            while True:
                block = response.read(1024 * 1024)
                if not block:
                    break
                out.write(block)
        temporary.replace(target)
    finally:
        if temporary.exists():
            temporary.unlink()


def stage_missing_files(
    path: str | os.PathLike | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the checkpoint). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them from {MODEL_SPEC['checkpoint_url']}"
        )
    fetch = downloader or _openmmlab_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def verify_runtime_versions() -> dict[str, str]:
    expected = {
        "mmdet": MODEL_SPEC["mmdetection_version"],
        "mmcv": MODEL_SPEC["mmcv_version"],
        "mmengine": MODEL_SPEC["mmengine_version"],
    }
    actual = {
        "mmdet": _version("mmdet"),
        "mmcv": _version("mmcv"),
        "mmengine": _version("mmengine"),
    }
    for name, expected_version in expected.items():
        if actual[name] != expected_version:
            raise RuntimeError(
                f"Unsupported {name} {actual[name]}; this runtime is qualified for {expected_version}."
            )
    return actual


def resolve_packaged_config() -> Path:
    import mmdet

    config = (
        Path(mmdet.__file__).resolve().parent
        / ".mim"
        / "configs"
        / MODEL_SPEC["config"]
    )
    if not config.is_file():
        raise RuntimeError(
            f"The pinned MMDetection package does not contain the expected config: {config}"
        )
    return config


def acquire_verified_checkpoint(cache_dir: str | os.PathLike = ".dimer-models") -> Path:
    root = Path(cache_dir).expanduser().resolve()
    root.mkdir(parents=True, exist_ok=True)
    target = root / Path(MODEL_SPEC["checkpoint_url"]).name

    def valid(path: Path) -> bool:
        return (
            path.is_file()
            and path.stat().st_size == MODEL_SPEC["checkpoint_size_bytes"]
            and _sha256(path) == MODEL_SPEC["checkpoint_sha256"]
        )

    if valid(target):
        return target
    if target.exists():
        target.unlink()

    fd, temporary_name = tempfile.mkstemp(prefix="dimer-swin-detection-", suffix=".pth", dir=root)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        with urllib.request.urlopen(MODEL_SPEC["checkpoint_url"], timeout=120) as response, temporary.open("wb") as out:
            while True:
                block = response.read(1024 * 1024)
                if not block:
                    break
                out.write(block)
        if temporary.stat().st_size != MODEL_SPEC["checkpoint_size_bytes"]:
            raise RuntimeError(
                f"Checkpoint size mismatch: got {temporary.stat().st_size}, expected {MODEL_SPEC['checkpoint_size_bytes']}."
            )
        digest = _sha256(temporary)
        if digest != MODEL_SPEC["checkpoint_sha256"]:
            raise RuntimeError(
                f"Checkpoint SHA-256 mismatch: got {digest}, expected {MODEL_SPEC['checkpoint_sha256']}."
            )
        temporary.replace(target)
    finally:
        if temporary.exists():
            temporary.unlink()
    return target


def validate_image(path: str | os.PathLike, *, max_pixels: int = 64_000_000) -> dict:
    image_path = Path(path).expanduser().resolve()
    if not image_path.is_file():
        raise ValueError(f"Image does not exist: {image_path}")
    try:
        with Image.open(image_path) as image:
            image.verify()
        with Image.open(image_path) as image:
            width, height = image.size
            mode = image.mode
    except Exception as exc:
        raise ValueError(f"Input is not a readable image: {image_path}: {exc}") from exc
    if width < 1 or height < 1:
        raise ValueError(f"Image dimensions must be positive; got {width}x{height}.")
    if width * height > max_pixels:
        raise ValueError(
            f"Image has {width * height:,} pixels; ceiling is {max_pixels:,}. Resize before inference."
        )
    return {"path": str(image_path), "width": width, "height": height, "mode": mode}


INPUT_SCHEMA: dict = {
    "input": "one or more image files readable by Pillow (any mode), given by path",
    "pixels": [1, MAX_PIXELS],
    "score_threshold": [0.0, 1.0],
    "max_detections": [1, None],
    "classes": "80 COCO 2017 categories in MMDetection order",
    "preprocessing": "MMDetection test pipeline of the pinned config (resize, normalise, pad); nothing is altered by this module",
}


def _check_request(score_threshold: float, max_detections: int) -> None:
    if not 0.0 <= score_threshold <= 1.0:
        raise ValueError("score_threshold must be between 0 and 1.")
    if max_detections < 1:
        raise ValueError("max_detections must be positive.")


def validate_inputs(
    images: str | os.PathLike | Iterable[str | os.PathLike],
    *,
    score_threshold: float = 0.0,
    max_detections: int = 300,
    names: Iterable[str] | None = None,
) -> dict:
    """Validation stage: return the input manifest (schema, per-image observations, request, verdict).

    Rejection is reported by raising exactly as ``predict`` would (``validate_image`` for each image,
    then the request ceilings); a caller that wants the finding recorded catches the exception and
    stores ``str(exc)`` under ``findings``.
    """
    _check_request(score_threshold, max_detections)
    paths = [images] if isinstance(images, (str, os.PathLike)) else list(images)
    observed = [validate_image(p) for p in paths]
    ids = list(names) if names is not None else [Path(o["path"]).name for o in observed]
    if len(ids) != len(observed):
        raise ValueError("names must have one entry per image")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": ids[i], **o} for i, o in enumerate(observed)],
        "score_threshold": score_threshold,
        "max_detections": max_detections,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    detections: Iterable[Detection | dict],
    ground_truth: Iterable[dict] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``ground_truth`` (one ``{"image_id", "boxes": [{"class_id", "bbox_xyxy"}]}`` entry per evaluated
    image) the report carries ``coco_box_ap`` — the repository's metric helper: COCO AP@[0.50:0.95],
    AP50 and AP75 via pycocotools — with the verdict ``sample-sanity`` and the empty-detector baseline
    (AP 0 by construction). Without it the verdict is ``not-measurable`` (EVAL9) and the report says
    what labelled data would make the task measurable; detection counts are sanity evidence that the
    inference path executed.
    """
    rows = [d.to_dict() if isinstance(d, Detection) else dict(d) for d in detections]
    images = sorted({row["image_id"] for row in rows})
    base = {
        "task": "COCO-80 object detection",
        "score_semantics": "MMDetection class confidence scores; uncalibrated, not probabilities of correctness",
        "decision_threshold": "caller-owned; score_threshold is an output filter, not a deployment threshold",
        "sample_kind": sample_kind,
        "n_images": len(images),
        "n_detections": len(rows),
        "context": {
            "upstream_reported_box_ap": MODEL_SPEC["upstream_reported_box_ap"],
            "note": "upstream full-COCO box AP as reported by OpenMMLab; not measured here",
        },
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if ground_truth is None:
        return {
            **base,
            "metrics": [],
            "baselines": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth boxes were supplied for the evaluated images",
            "needs": (
                "ground-truth boxes (class id + xyxy) for the evaluated images, scored with coco_box_ap "
                "(pycocotools COCO AP@[0.50:0.95], AP50, AP75) against the empty-detector baseline; "
                "a labelled set from the deployment domain for any generalisable claim"
            ),
        }
    ap = coco_box_ap(rows, ground_truth)
    estimation = f"single labelled sample of {ap['n_images']} image(s) / {ap['n_ground_truth_boxes']} boxes, no dispersion estimate"
    return {
        **base,
        "n_images": ap["n_images"],
        "metrics": [
            {"id": "coco_box_ap", "iou": "0.50:0.95", "value": ap["coco_ap_50_95"], "estimation": estimation},
            {"id": "coco_box_ap", "iou": "0.50", "value": ap["ap50"], "estimation": estimation},
            {"id": "coco_box_ap", "iou": "0.75", "value": ap["ap75"], "estimation": estimation},
        ],
        "baselines": [
            {"id": "empty_detector", "coco_box_ap": 0.0, "note": "a detector returning no boxes scores AP 0 by construction"}
        ],
        "verdict": "sample-sanity",
        "reason": f"{ap['n_images']} labelled image(s) with {ap['n_ground_truth_boxes']} ground-truth boxes from the tutorial sample; not a benchmark",
        "needs": "a representative labelled holdout from the deployment domain for any generalisable AP claim",
    }


class DimerSwinDetector:
    """Public task-inference API for the pinned DIMER Swin object detector.

    The upstream `.pth` checkpoint is a code-capable PyTorch serialization. This
    runtime verifies its size and SHA-256 before MMDetection deserializes it, but
    digest verification establishes byte identity, not author authenticity.
    Only use the checkpoint when the pinned OpenMMLab source is trusted.
    """

    def __init__(
        self,
        *,
        cache_dir: str | os.PathLike = ".dimer-models",
        device: str = "cpu",
        checkpoint: str | os.PathLike | None = None,
        source: str = "openmmlab-cache",
    ):
        versions = verify_runtime_versions()
        checkpoint = Path(checkpoint) if checkpoint is not None else acquire_verified_checkpoint(cache_dir)
        config = resolve_packaged_config()
        from mmdet.apis import init_detector

        self.model = init_detector(str(config), str(checkpoint), device=device)
        self.device = device
        self.checkpoint = checkpoint
        self.config = config
        self.versions = versions
        self.source = source
        self.classes = tuple(self.model.dataset_meta.get("classes", ()))
        if not self.classes:
            raise RuntimeError("MMDetection model did not expose its class ordering.")

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str = "cpu",
        weights_dir: str | os.PathLike | None = None,
        allow_download: bool = False,
    ) -> DimerSwinDetector:
        """Load from the fleet snapshot directory: stage absent manifest entries (only with
        ``allow_download=True``, from the pinned OpenMMLab URL), re-hash every entry against the
        manifest and MODEL_SPEC, then deserialise the checkpoint through the pinned OpenMMLab loader.
        The ``.pth`` is code-capable PyTorch serialization: digest verification fixes the bytes, not
        the author — see the class docstring.
        """
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        if not (root / MANIFEST_NAME).is_file():
            raise FileNotFoundError(
                f"no snapshot manifest at {root}; use DimerSwinDetector(cache_dir=...) for the cache path"
            )
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        return cls(device=device, checkpoint=root / WEIGHTS_FILE, source="local-snapshot")

    def predict(
        self,
        image: str | os.PathLike,
        *,
        score_threshold: float = 0.0,
        max_detections: int = 300,
    ) -> list[Detection]:
        if not 0.0 <= score_threshold <= 1.0:
            raise ValueError("score_threshold must be between 0 and 1.")
        if max_detections < 1:
            raise ValueError("max_detections must be positive.")
        info = validate_image(image)
        from mmdet.apis import inference_detector

        sample = inference_detector(self.model, info["path"])
        instances = sample.pred_instances.cpu()
        detections: list[Detection] = []
        for bbox, score, label in zip(instances.bboxes, instances.scores, instances.labels, strict=False):
            score_value = float(score.item())
            if score_value < score_threshold:
                continue
            label_value = int(label.item())
            coords = tuple(float(v) for v in bbox.tolist())
            detections.append(
                Detection(
                    image_id=Path(info["path"]).name,
                    class_id=label_value,
                    class_name=self.classes[label_value],
                    score=score_value,
                    bbox_xyxy=coords,
                )
            )
        detections.sort(key=lambda d: d.score, reverse=True)
        return detections[:max_detections]

    def predict_many(self, images: Iterable[str | os.PathLike], **kwargs) -> list[Detection]:
        output: list[Detection] = []
        for image in images:
            output.extend(self.predict(image, **kwargs))
        return output

    def provenance(self) -> dict:
        import platform

        import torch

        return {
            "runtime": MODEL_SPEC,
            "effective": {
                "python": platform.python_version(),
                "torch": torch.__version__,
                **self.versions,
                "device": self.device,
                "source": self.source,
                "classes": list(self.classes),
                "checkpoint_path": str(self.checkpoint),
                "checkpoint_sha256": _sha256(self.checkpoint),
            },
            "score_semantics": "MMDetection class confidence scores; uncalibrated, not probabilities of correctness.",
            "decision_threshold": "caller-owned; score_threshold is an output filter, not a universal deployment threshold.",
        }

    def write_provenance(self, path: str | os.PathLike) -> Path:
        target = Path(path)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(json.dumps(self.provenance(), indent=2) + "\n")
        return target

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `1`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the OpenMMLab checkpoint host (`download.openmmlab.com`) **at MMDetection release-tag commit `44ebd17b145c…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `DimerSwinDetector.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "swin-t-mask-rcnn-coco",
  "modelId": "open-mmlab/mmdetection:mask-rcnn_swin-t-p4-w7_fpn_1x_coco",
  "revision": "44ebd17b145c2372c4b700bfb9cb20dbd28ab64a",
  "files": [
    {
      "path": "mask_rcnn_swin-t-p4-w7_fpn_1x_coco_20210902_120937-9d6b7cfa.pth",
      "bytes": 191461353,
      "sha256": "9d6b7cfaa4aad52ef559611bea454f01d6f1f17c82a1abfac0d71631a193a291"
    }
  ],
  "totalBytes": 191461353
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = DimerSwinDetector.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Confirm the qualified runtime

The carried package fails closed on version drift: `verify_runtime_versions` (called when the model was constructed above) compares the installed `mmdet`, `mmcv` and `mmengine` distributions with the versions pinned in `MODEL_SPEC`, and this cell additionally asserts the Python 3.10 interpreter the OpenMMLab wheels were built for. Look for a dictionary reporting Python 3.10.x, `torch` 2.1.2+cpu, MMDetection 3.3.0, MMCV 2.1.0, MMEngine 0.10.7, 80 classes, the verified checkpoint file name and the `local-snapshot` source.

In [ ]:
import sys

if sys.version_info[:2] != (3, 10):
    raise RuntimeError(f'Python 3.10 is required by the qualified OpenMMLab runtime (see Prerequisites); this kernel is {sys.version.split()[0]}. Use a Python 3.10 kernel.')
print({'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, **pipe.versions, 'classes': len(pipe.classes), 'checkpoint': pipe.checkpoint.name, 'source': pipe.source, 'device': pipe.device})

## 5. Generate the synthetic sample, or opt into COCO8 / BYOD

The default sample is **synthetic**: a deterministic 640×480 scene (a red→green gradient background with three flat-coloured shapes) drawn in code and written to `sample/`, so it needs no download and its SHA-256 is printed for the record. It depicts no COCO object, so it has **no ground truth**: whatever the detector returns is a sanity check that the input contract, preprocessing and forward pass work, not a correctness measurement. Two gates are off by default. `USE_COCO8` fetches the public COCO8 archive from its pinned release URL, refuses it unless its SHA-256 equals the recorded digest, extracts only the four validation images and their YOLO-format labels member by member after path and size checks (no `extractall`), and converts the labels to ground-truth boxes with the carried `boxes_from_yolo_labels` — the notebook does not resplit or relabel anything. `USE_BYOD` uploads one image; BYOD has no ground truth unless you build it yourself. Look for a dictionary naming the sample kind, the image files, their digests and whether ground truth exists.

In [ ]:
import hashlib
import stat
import zipfile
from pathlib import Path, PurePosixPath

from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
USE_COCO8 = False  # @param {type:"boolean"}
SCORE_THRESHOLD = 0.0  # evaluation sees the full detector output; a display cut-off is the caller's choice
MAX_DETECTIONS = 300
COCO8_URL = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip'
COCO8_SHA256 = '54c67fe9ef88313e021ec0e92b73c200167bb0a86633e8df8658d832cca828c9'
COCO8_MAX_EXPANDED_BYTES = 25 * 1024 * 1024
sample_dir = Path('sample')
sample_dir.mkdir(exist_ok=True)
ground_truth = None
if USE_BYOD and USE_COCO8:
    raise ValueError('Enable at most one of USE_BYOD and USE_COCO8.')
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    upload_name = next(iter(uploaded))
    image_path = sample_dir / Path(upload_name).name
    image_path.write_bytes(uploaded[upload_name])
    image_paths = [image_path]
    sample_kind = 'BYOD'
elif USE_COCO8:
    import urllib.request

    archive = sample_dir / 'coco8.zip'
    if not archive.exists():
        urllib.request.urlretrieve(COCO8_URL, archive)
    archive_sha256 = hashlib.sha256(archive.read_bytes()).hexdigest()
    if archive_sha256 != COCO8_SHA256:
        raise RuntimeError(f'COCO8 archive digest {archive_sha256} != pinned {COCO8_SHA256}; refusing to extract')
    with zipfile.ZipFile(archive) as zf:
        members = zf.infolist()
        expanded = 0
        for info in members:
            member = PurePosixPath(info.filename)
            if member.is_absolute() or '..' in member.parts or '\\' in info.filename:
                raise ValueError(f'unsafe archive member: {info.filename}')
            if stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError(f'symlink refused: {info.filename}')
            expanded += info.file_size
            if expanded > COCO8_MAX_EXPANDED_BYTES:
                raise ValueError('archive exceeds the expanded-size ceiling')
        for info in members:
            if info.filename.startswith(('coco8/images/val/', 'coco8/labels/val/')) and not info.is_dir():
                zf.extract(info, sample_dir)
    coco8 = sample_dir / 'coco8'
    image_paths = sorted((coco8 / 'images' / 'val').glob('*.jpg'))
    if not image_paths:
        raise RuntimeError('COCO8 validation images were not found after extraction.')
    ground_truth = []
    for path in image_paths:
        with Image.open(path) as opened:
            width, height = opened.size
        label_text = (coco8 / 'labels' / 'val' / f'{path.stem}.txt').read_text(encoding='utf-8')
        ground_truth.append({'image_id': path.name, 'boxes': boxes_from_yolo_labels(label_text, width, height)})
    sample_kind = 'COCO8-val'
else:
    # Deterministic synthetic scene: no randomness, so no seed is needed and the digest is stable.
    width, height = 640, 480
    ramp = numpy.linspace(0.0, 255.0, width)
    red = numpy.tile(ramp, (height, 1))
    green = numpy.tile(numpy.linspace(0.0, 255.0, height)[:, None], (1, width))
    blue = (red + green) / 2.0
    array = numpy.rint(numpy.stack([red, green, blue], axis=-1)).astype(numpy.uint8)
    scene = Image.fromarray(array, mode='RGB')
    draw = ImageDraw.Draw(scene)
    draw.rectangle([60, 300, 260, 440], fill=(20, 20, 20))
    draw.ellipse([380, 80, 560, 260], fill=(240, 240, 240))
    draw.polygon([(320, 460), (400, 330), (480, 460)], fill=(30, 90, 200))
    image_path = sample_dir / 'synthetic_scene_640x480.png'
    scene.save(image_path)
    image_paths = [image_path]
    sample_kind = 'synthetic'
image_names = [path.name for path in image_paths]
sample_sha256 = {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in image_paths}
n_ground_truth_boxes = None if ground_truth is None else sum(len(entry['boxes']) for entry in ground_truth)
print({'sample_kind': sample_kind, 'images': image_names, 'sha256': sample_sha256, 'ground_truth_boxes': n_ground_truth_boxes})

## 6. Validate the input → input manifest

`validate_inputs` is the package's public validation stage: it applies exactly the checks `predict` applies — the request ceilings (`score_threshold` in 0..1, `max_detections` ≥ 1) and, per image, `validate_image` (the file exists, Pillow can decode it, positive dimensions, at most `MAX_PIXELS` = 64,000,000 pixels) — and returns an **input manifest** naming the schema and ceilings, each input's observed path, size and mode, the request parameters and the verdict. The manifest is written to `outputs/swin_detection_task_inference_input_manifest.json`. To show what rejection looks like, the cell also validates a path that does not exist and records the package's own error message as a finding. Inside the package every accepted image goes through the pinned config's MMDetection test pipeline (resize, normalise, pad); nothing is dropped or altered by the package itself.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MAX_PIXELS': MAX_PIXELS, 'score_threshold': INPUT_SCHEMA['score_threshold'], 'max_detections': INPUT_SCHEMA['max_detections'], 'classes': len(pipe.classes)}})
input_manifest = validate_inputs(image_paths, score_threshold=SCORE_THRESHOLD, max_detections=MAX_DETECTIONS, names=image_names)
# Demonstrate rejection on an input that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs(sample_dir / 'does-not-exist.png')
except ValueError as exc:
    input_manifest['findings'].append({'input': 'missing-file-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/swin_detection_task_inference_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 7. Detect

`predict_many` runs `predict` per image through the pinned MMDetection inference path and returns `Detection` records (`image_id`, `class_id`, `class_name`, `score`, `bbox_xyxy` in pixel coordinates), **ordered by descending score within each image** and cut at `max_detections`. The `score` is the MMDetection class confidence after the config's own NMS: it is **uncalibrated**, not a probability that the box is correct, and the package ships no deployment threshold — `score_threshold` is an output filter the caller owns (0.0 here so evaluation sees the full output). Inference is deterministic given the same weights, device and library versions (`model.eval()`, no sampling); CPU kernel choices can reorder near-tied scores. Look for the per-image detection counts and the five highest-scoring boxes; on the synthetic scene expect few or low-scoring detections.

In [ ]:
detections = pipe.predict_many(image_paths, score_threshold=SCORE_THRESHOLD, max_detections=MAX_DETECTIONS)
rows = [detection.to_dict() for detection in detections]
counts = {name: sum(1 for row in rows if row['image_id'] == name) for name in image_names}
print({'n_detections': len(rows), 'per_image': counts, 'score_threshold': SCORE_THRESHOLD, 'max_detections': MAX_DETECTIONS})
for rank, row in enumerate(sorted(rows, key=lambda row: row['score'], reverse=True)[:5], start=1):
    print(f"{rank:>2}. {row['image_id']:<24} class {row['class_id']:>2} {row['class_name']:<14} score {row['score']:.4f}  bbox_xyxy {[round(v, 1) for v in row['bbox_xyxy']]}")

## 8. Evaluate → evaluation report

`evaluation_report` is the package's public evaluation stage and always produces a report. When ground-truth boxes exist (the `USE_COCO8` path) it carries `coco_box_ap` — the repository's metric helper, COCO AP@[0.50:0.95], AP50 and AP75 via `pycocotools` over axis-aligned boxes — with the verdict `sample-sanity` and the empty-detector baseline (AP 0 by construction): a four-image tutorial metric with high sampling variance and no dispersion estimate, not comparable to the upstream full-COCO box AP of 42.7 that `MODEL_SPEC` records as upstream-reported context. On the synthetic default sample (and on BYOD without labels) no metric exists, so the verdict is `not-measurable` and the report states what would make the task measurable: ground-truth boxes for the evaluated images, or a labelled holdout from the deployment domain. The report is written to `outputs/swin_detection_task_inference_evaluation_report.json`.

In [ ]:
report = evaluation_report(detections, ground_truth, sample_kind=sample_kind)
with open('outputs/swin_detection_task_inference_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No ground-truth boxes were supplied, so coco_box_ap is not computed; the detections above are sanity evidence only.')

## 9. Export outputs and provenance

Machine-readable JSON preserves every detection (score-ordered per image), the evaluation report, the input manifest, the sample identity and digests, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable revision label, the checkpoint digest the package verified, and the runtime identity (Python, `torch`, `mmdet`, `mmcv`, `mmengine`, device). The detections are also written as CSV with explicit `rank`, `class_id`, `class_name`, `score` and `x1,y1,x2,y2` columns so score ordering and pixel coordinates survive downstream use. No credentials are recorded.

In [ ]:
import csv

payload = {
    'detections': rows,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'images': image_names, 'sha256': sample_sha256, 'ground_truth_boxes': n_ground_truth_boxes},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'checkpoint_sha256': MODEL_SPEC['checkpoint_sha256'],
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'numpy': numpy.__version__,
        **pipe.versions,
        'device': pipe.device,
    },
}
with open('outputs/swin_detection_task_inference_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/swin_detection_task_inference_detections.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image_id', 'rank', 'class_id', 'class_name', 'score', 'x1', 'y1', 'x2', 'y2'])
    for name in image_names:
        for rank, row in enumerate([row for row in rows if row['image_id'] == name], start=1):
            writer.writerow([name, rank, row['class_id'], row['class_name'], f"{row['score']:.6f}", *[f'{v:.2f}' for v in row['bbox_xyxy']]])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

Each detection is a class from the fixed 80-category COCO 2017 label space with an axis-aligned box and an **uncalibrated** MMDetection score; the package ships no acceptance threshold and `score_threshold` is an output filter the caller owns. On the synthetic scene the detections are meaningless by construction and the evaluation report says `not-measurable`; a `coco_box_ap` value from the four-image COCO8 subset is tutorial evidence for those images and must not be generalized to a domain, camera, object size distribution or class mix. Objects outside the COCO categories, crowded or tiny objects, unusual viewpoints, and domain shifts (medical, aerial, line art) all degrade results in ways the package does not detect. The package provides no segmentation, tracking, keypoint, classification, or training capability, and the `.pth` checkpoint remains a code-capable serialization whose digest check fixes the bytes, not the author.

Successful execution proves that the recorded repository revision's package, carried in this notebook, can acquire and digest-verify the pinned OpenMMLab checkpoint, assert the qualified Python 3.10 / MMDetection 3.3.0 runtime, validate the demonstrated input, execute the public detection path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, reproduction of the upstream COCO result, score calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** enable `USE_COCO8` to see the report switch to `sample-sanity` with `coco_box_ap` at IoU 0.50:0.95, 0.50 and 0.75 against the empty-detector baseline; enable `USE_BYOD` with a photograph from your own domain and inspect the score distribution before choosing a display threshold; label a small holdout from that domain in the `boxes_from_yolo_labels` format and compare its AP with the COCO8 value.

## References

- Repository README: https://github.com/kurtvalcorza/swin-detection-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/swin-detection-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/swin-detection-pipeline/blob/main/docs/WEIGHTS.md
- Pinned checkpoint (OpenMMLab host): https://download.openmmlab.com/mmdetection/v2.0/swin/mask_rcnn_swin-t-p4-w7_fpn_1x_coco/mask_rcnn_swin-t-p4-w7_fpn_1x_coco_20210902_120937-9d6b7cfa.pth
- Config source (MMDetection v3.3.0, `configs/swin`): https://github.com/open-mmlab/mmdetection/tree/v3.3.0/configs/swin
- COCO8 sample (Ultralytics assets release): https://github.com/ultralytics/assets/releases/tag/v0.0.0
- Upstream project: https://github.com/microsoft/Swin-Transformer
- Swin Transformer: Hierarchical Vision Transformer using Shifted Windows: https://arxiv.org/abs/2103.14030
- Mask R-CNN: https://arxiv.org/abs/1703.06870
- Microsoft COCO: Common Objects in Context: https://arxiv.org/abs/1405.0312